# SBERT scoring

Get a score for each resume + job pair using sentence-bert. Higher = closer.

In [ ]:
!pip install sentence-transformers pandas scikit-learn

In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [ ]:
# upload jobs.csv and resume_variants.csv
from google.colab import files
up = files.upload()
for f in up:
    os.rename(f, f"data/{f}")

In [ ]:
jobs = pd.read_csv("data/jobs.csv")
res = pd.read_csv("data/resume_variants.csv")
print(jobs.shape, res.shape)

In [ ]:
# stick the job text together so we can embed it
jobs["job_text"] = jobs["title"] + " " + jobs["domain"] + " " + jobs["company_name"] + " " + jobs["job_description"]

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# embed everything once
job_emb = model.encode(jobs["job_text"].tolist())
res_emb = model.encode(res["resume_text"].tolist())
print(res_emb.shape, job_emb.shape)

In [ ]:
out = []
for i in range(len(res)):
    for j in range(len(jobs)):
        s = float(cosine_similarity(res_emb[i:i+1], job_emb[j:j+1])[0][0])
        out.append({
            "resume_id": res.iloc[i]["resume_id"],
            "version": res.iloc[i]["version"],
            "changed_signal": res.iloc[i]["changed_signal"],
            "job_id": jobs.iloc[j]["job_id"],
            "job_title": jobs.iloc[j]["title"],
            "similarity_score": s,
        })

df = pd.DataFrame(out)
print(len(df), "pairs")
df.head()

In [ ]:
df.to_csv("results/sbert_scores.csv", index=False)
files.download("results/sbert_scores.csv")